<a href="https://colab.research.google.com/github/fatemarampurawalaa-sudo/LLM-practice/blob/main/Langchain_llamaindex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain-google-genai langchain-core

import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("Setup done ✅")



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
Setup done ✅


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatGoogleGenerativeAI(model="gemini-flash-lite-latest")

template = ChatPromptTemplate.from_template(
    "Write a 2-sentence listing for a {beds}-bed apartment in {area} priced at AED {price}."
)

chain = template | model | StrOutputParser()

print(chain.invoke({"beds": 2, "area": "Dubai Marina", "price": "1,200,000"}))

Experience waterfront living in this stunning 2-bedroom apartment in Dubai Marina, offered at an incredible value of AED 1,200,000. Enjoy modern interiors, breathtaking marina views, and unbeatable proximity to world-class dining, retail, and the beach.


In [3]:
!pip install -q langchain

In [4]:
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def calculate_yield(price: float, annual_rent: float) -> str:
    """Calculate the gross rental yield of a property. Price and rent are in AED."""
    print(f"🔧 Tool used: calculate_yield(price={price}, annual_rent={annual_rent})")
    return f"{annual_rent / price * 100:.2f}%"

agent = create_agent(model, tools=[calculate_yield])

result = agent.invoke({"messages": [{"role": "user", "content": "What's the yield on a 1.2M flat renting for 85k?"}]})
print(result["messages"][-1].text)

🔧 Tool used: calculate_yield(price=1200000.0, annual_rent=85000.0)
The gross rental yield on a 1,200,000 AED flat renting for 85,000 AED per year is **7.08%**.


In [5]:
!pip install -q llama-index-core llama-index-llms-google-genai llama-index-embeddings-google-genai

In [7]:
import nest_asyncio
nest_asyncio.apply()

In [8]:
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

key = os.environ["GOOGLE_API_KEY"]
Settings.llm = GoogleGenAI(model="gemini-flash-lite-latest", api_key=key)
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001", api_key=key)

lease = Document(text="""Apartment 1204, Marina Heights, Dubai Marina. Tenant: Priya Sharma. Annual rent AED 95,000 in 4 cheques.
One covered parking space, number B2-118, is included. Pets are not allowed.
The tenant pays DEWA bills. Chiller charges are included in the rent.""")

index = VectorStoreIndex.from_documents([lease])
query_engine = index.as_query_engine()

print(query_engine.query("Is parking included?"))

Yes, one covered parking space (number B2-118) is included.
